# tokenizer

In [1]:
"""Custom tokenizer for RHM (Random Hierarchy Model) training and evaluation."""

import typing as t
from pathlib import Path

from transformers import PreTrainedTokenizer


class RHMTokenizer(PreTrainedTokenizer):
    """Custom tokenizer for RHM with direct integer-to-token mapping."""

    def __init__(
        self,
        vocab_size: int = 37,
        pad_token: str = "<pad>",
        eos_token: str = "<eos>",
        sep_token: str = "<sep>",
        mask_token: str = "<mask>",
        unk_token: str = "<unk>",
        **kwargs,
    ):
        """Initialize RHM tokenizer with controlled vocabulary.
        
        Args:
            vocab_size: Size of RHM vocabulary (default 37)
            pad_token: Padding token string
            eos_token: End of sequence token string
            sep_token: Separator token string
            mask_token: Mask token for MLM
            unk_token: Unknown token string
        """
        # Store vocab size in private attribute to avoid property conflict
        self._rhm_vocab_size = vocab_size
        
        # Create vocabulary: integers + special tokens
        self._vocab = {}
        self._ids_to_tokens = {}
        
        # Add integer tokens (0 to vocab_size-1)
        for i in range(vocab_size):
            token = str(i)
            self._vocab[token] = i
            self._ids_to_tokens[i] = token
        
        # Add special tokens
        special_tokens = {
            pad_token: vocab_size,
            eos_token: vocab_size + 1,
            sep_token: vocab_size + 2,
            mask_token: vocab_size + 3,
            unk_token: vocab_size + 4,
        }
        
        for token, token_id in special_tokens.items():
            self._vocab[token] = token_id
            self._ids_to_tokens[token_id] = token
        
        # Initialize parent class
        super().__init__(
            pad_token=pad_token,
            eos_token=eos_token,
            sep_token=sep_token,
            mask_token=mask_token,
            unk_token=unk_token,
            **kwargs,
        )

    @property
    def vocab(self) -> dict[str, int]:
        """Return vocabulary mapping."""
        return self._vocab

    @property
    def vocab_size(self) -> int:
        """Return vocabulary size including special tokens."""
        return len(self._vocab)

    @property
    def rhm_vocab_size(self) -> int:
        """Return original RHM vocabulary size (without special tokens)."""
        return self._rhm_vocab_size

    def get_vocab(self) -> dict[str, int]:
        """Get vocabulary for compatibility."""
        return self._vocab

    def _tokenize(self, text: str) -> list[str]:
        """Tokenize text into tokens."""
        # Split by whitespace and convert each part
        tokens = []
        for part in text.strip().split():
            if part in self._vocab:
                tokens.append(part)
            else:
                # Try to parse as integer
                try:
                    int_val = int(part)
                    if 0 <= int_val < self._rhm_vocab_size:
                        tokens.append(str(int_val))
                    else:
                        tokens.append(self.unk_token)
                except ValueError:
                    tokens.append(self.unk_token)
        return tokens

    def _convert_token_to_id(self, token: str) -> int:
        """Convert token to ID."""
        return self._vocab.get(token, self._vocab[self.unk_token])

    def _convert_id_to_token(self, index: int) -> str:
        """Convert ID to token."""
        return self._ids_to_tokens.get(index, self.unk_token)

    def convert_tokens_to_string(self, tokens: list[str]) -> str:
        """Convert tokens back to string."""
        return " ".join(tokens)

    def encode_sequence(self, sequence: list[int]) -> list[int]:
        """Encode RHM integer sequence directly to token IDs."""
        token_ids = []
        for val in sequence:
            if 0 <= val < self._rhm_vocab_size:
                token_ids.append(val)  # Direct mapping for integers
            else:
                token_ids.append(self._vocab[self.unk_token])
        return token_ids

    def decode_sequence(self, token_ids: list[int]) -> list[int]:
        """Decode token IDs back to RHM integer sequence."""
        sequence = []
        for token_id in token_ids:
            if 0 <= token_id < self._rhm_vocab_size:
                sequence.append(token_id)  # Direct mapping for integers
            # Skip special tokens in decoded sequence
        return sequence

    def add_special_tokens_to_sequence(
        self, sequence: list[int], add_eos: bool = True, add_sep: bool = False
    ) -> list[int]:
        """Add special tokens to RHM sequence."""
        result = sequence.copy()
        
        if add_sep:
            result.append(self.sep_token_id)
        
        if add_eos:
            result.append(self.eos_token_id)
            
        return result

    def format_icl_prompt(
        self, 
        examples: list[tuple[list[int], list[int]]], 
        query: list[int]
    ) -> list[int]:
        """Format in-context learning prompt using RHM sequences."""
        prompt = []
        
        # Add examples
        for input_seq, output_seq in examples:
            prompt.extend(input_seq)
            prompt.append(self.sep_token_id)  # Separator between input and output
            prompt.extend(output_seq)
            prompt.append(self.eos_token_id)  # End of example
        
        # Add query
        prompt.extend(query)
        prompt.append(self.sep_token_id)  # Indicate start of expected output
        
        return prompt

    def save_vocabulary(self, save_directory: str | Path, filename_prefix: str | None = None) -> tuple[str]:
        """Save vocabulary to file."""
        save_directory = Path(save_directory)
        save_directory.mkdir(parents=True, exist_ok=True)
        
        if filename_prefix is None:
            filename_prefix = "vocab"
        
        vocab_file = save_directory / f"{filename_prefix}.json"
        
        import json
        with vocab_file.open("w", encoding="utf-8") as f:
            json.dump(self._vocab, f, indent=2)
        
        return (str(vocab_file),)



In [2]:

def test_rhm_tokenizer() -> None:
    """Test function for RHM tokenizer."""
    tokenizer = RHMTokenizer(vocab_size=37)
    
    # Test basic functionality
    print("=== RHM Tokenizer Test ===")
    print(f"Vocab size: {tokenizer.vocab_size}")
    print(f"Pad token ID: {tokenizer.pad_token_id}")
    print(f"EOS token ID: {tokenizer.eos_token_id}")
    print(f"Sep token ID: {tokenizer.sep_token_id}")
    print(f"Mask token ID: {tokenizer.mask_token_id}")
    
    # Test sequence encoding
    test_sequence = [5, 2, 8, 1, 9]
    encoded = tokenizer.encode_sequence(test_sequence)
    decoded = tokenizer.decode_sequence(encoded)
    print(f"\nOriginal sequence: {test_sequence}")
    print(f"Encoded: {encoded}")
    print(f"Decoded: {decoded}")
    
    # Test ICL formatting
    examples = [([1, 2], [3, 4]), ([5, 6], [7, 8])]
    query = [9, 10]
    icl_prompt = tokenizer.format_icl_prompt(examples, query)
    print(f"\nICL prompt: {icl_prompt}")
    
    # Test text tokenization
    text = "5 2 8 <sep> 1 9"
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    print(f"\nText: '{text}'")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {token_ids}")


if __name__ == "__main__":
    test_rhm_tokenizer()

=== RHM Tokenizer Test ===
Vocab size: 42
Pad token ID: 37
EOS token ID: 38
Sep token ID: 39
Mask token ID: 40

Original sequence: [5, 2, 8, 1, 9]
Encoded: [5, 2, 8, 1, 9]
Decoded: [5, 2, 8, 1, 9]

ICL prompt: [1, 2, 39, 3, 4, 38, 5, 6, 39, 7, 8, 38, 9, 10, 39]

Text: '5 2 8 <sep> 1 9'
Tokens: ['5', '2', '8', '<sep>', '1', '9']
Token IDs: [5, 2, 8, 39, 1, 9]


# train config


In [ ]:
"""RHM Training Configuration with Custom Tokenizer Support"""

import logging
import random
import typing as t
from dataclasses import dataclass, field
import os

from datasets import Dataset
from transformers import TrainingArguments


# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


@dataclass
class RHMTrainingConfig:
    """Configuration for RHM training with custom tokenizer support."""

    # Model configuration
    model_name_or_path: str | None = None
    vocab_size: int = 37  # RHM vocabulary size
    hidden_size: int = 512
    num_hidden_layers: int = 6
    num_attention_heads: int = 8
    intermediate_size: int = 2048
    max_position_embeddings: int = 2048

    # Tokenizer configuration
    pad_token: str = "<pad>"
    eos_token: str = "<eos>"
    sep_token: str = "<sep>"
    mask_token: str = "<mask>"
    unk_token: str = "<unk>"

    # Training configuration
    task_name: str = "mlm"  # "mlm" or "clm"
    output_dir: str = "./rhm_training_output"
    num_train_epochs: int = 10
    per_device_train_batch_size: int = 16
    per_device_eval_batch_size: int = 32
    gradient_accumulation_steps: int = 1
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    lr_scheduler_type: str = "linear"

    # Add these special token configuration fields:
    pad_token_id: int = 0
    mask_token_id: int = vocab_size + 1
    cls_token_id: int = vocab_size + 2
    sep_token_id: int = vocab_size + 3

    # Sequence processing
    max_sequence_length: int = 512
    pack_sequences: bool = True
    mlm: bool = True
    mlm_probability: float = 0.15

    # Checkpoint configuration
    save_strategy: str = "steps"
    save_steps: int = 500
    save_total_limit: int = 5
    load_best_model_at_end: bool = True
    metric_for_best_model: str = "eval_loss"
    greater_is_better: bool = False

    # Evaluation configuration
    evaluation_strategy: str = "steps"
    eval_steps: int = 500
    eval_accumulation_steps: int | None = None

    # Logging configuration
    logging_strategy: str = "steps"
    logging_steps: int = 100
    report_to: list[str] = field(default_factory=lambda: ["tensorboard"])
    run_name: str | None = None

    # Optimization configuration
    adam_beta1: float = 0.9
    adam_beta2: float = 0.999
    adam_epsilon: float = 1e-8
    max_grad_norm: float = 1.0

    # Early stopping
    early_stopping: bool = True
    early_stopping_patience: int = 3
    early_stopping_threshold: float = 0.0

    # Mixed precision
    fp16: bool = False
    bf16: bool = False

    # Data configuration
    dataloader_num_workers: int = 0
    dataloader_pin_memory: bool = True
    remove_unused_columns: bool = True

    # Dataset configuration
    dataset_path: str | None = None
    train_split_ratio: float = 0.8
    filter_config_L: int | None = None
    filter_config_m: int | None = None
    max_samples: int | None = None

    # Hierarchical analysis
    track_hierarchical_metrics: bool = True
    hierarchical_eval_frequency: int = 1000

    # Reproducibility
    seed: int = 42

    def create_tokenizer(self) -> RHMTokenizer:
        """Create RHM tokenizer with current configuration."""
        return RHMTokenizer(
            vocab_size=self.vocab_size,
            pad_token=self.pad_token,
            eos_token=self.eos_token,
            sep_token=self.sep_token,
            mask_token=self.mask_token,
            unk_token=self.unk_token,
        )

    @property
    def effective_vocab_size(self) -> int:
        """Get effective vocabulary size including special tokens."""
        return self.vocab_size + 5  # RHM tokens + 5 special tokens

    
    def to_training_arguments(self) -> TrainingArguments:
        """Convert to HuggingFace TrainingArguments with version compatibility."""
        training_args_dict = {
            "output_dir": self.output_dir,

            "output_dir": self.output_dir,
            "save_strategy": "steps",  # or "epoch"
            "save_steps": 500,  # adjust as needed
            "save_total_limit": 3,  # keep only last 3 checkpoints
            "load_best_model_at_end": True,
            "num_train_epochs": self.num_train_epochs,
            "per_device_train_batch_size": self.per_device_train_batch_size,
            "per_device_eval_batch_size": self.per_device_eval_batch_size,
            "gradient_accumulation_steps": self.gradient_accumulation_steps,
            "learning_rate": self.learning_rate,
            "weight_decay": self.weight_decay,
            "warmup_ratio": self.warmup_ratio,
            "lr_scheduler_type": self.lr_scheduler_type,
            "save_strategy": self.save_strategy,
            "save_steps": self.save_steps,
            "save_total_limit": self.save_total_limit,
            "load_best_model_at_end": self.load_best_model_at_end,
            "metric_for_best_model": self.metric_for_best_model,
            "greater_is_better": self.greater_is_better,
            "evaluation_strategy": self.evaluation_strategy,
            "eval_steps": self.eval_steps,
            "eval_accumulation_steps": self.eval_accumulation_steps,
            "logging_strategy": self.logging_strategy,
            "logging_steps": self.logging_steps,
            "report_to": self.report_to,
            "run_name": self.run_name,
            "adam_beta1": self.adam_beta1,
            "adam_beta2": self.adam_beta2,
            "adam_epsilon": self.adam_epsilon,
            "max_grad_norm": self.max_grad_norm,
            "fp16": self.fp16,
            "bf16": self.bf16,
            "dataloader_num_workers": self.dataloader_num_workers,
            "dataloader_pin_memory": self.dataloader_pin_memory,
            "remove_unused_columns": self.remove_unused_columns,
            "seed": self.seed,
        }
        if "wandb" in self.report_to:
            os.environ["WANDB_DIR"] = self.output_dir

        # Try creating TrainingArguments, remove problematic parameters if they fail
        try:
            return TrainingArguments(**training_args_dict)
        except TypeError as e:
            logger.warning(f"TrainingArguments creation failed: {e}")
            logger.info("Trying with minimal parameters for version compatibility...")
            
            # Minimal set of parameters that should work across versions
            minimal_args = {
                "output_dir": self.output_dir,
                "num_train_epochs": self.num_train_epochs,
                "per_device_train_batch_size": self.per_device_train_batch_size,
                "per_device_eval_batch_size": self.per_device_eval_batch_size,
                "learning_rate": self.learning_rate,
                "evaluation_strategy": self.evaluation_strategy,
                "eval_steps": self.eval_steps,
                "save_steps": self.save_steps,
                "logging_steps": self.logging_steps,
                "seed": self.seed,
            }
            
            return TrainingArguments(**minimal_args)





def prepare_packed_dataset(
    dataset_path: str,
    tokenizer: RHMTokenizer,
    config: RHMTrainingConfig,
    train_split_ratio: float = 0.8,
    filter_config_L: int | None = None,
    filter_config_m: int | None = None,
    max_samples: int | None = None,
) -> tuple[Dataset, Dataset, dict[str, t.Any]]:
    """Prepare packed dataset for HuggingFace training with custom tokenizer.

    Args:
        dataset_path: Path to unified RHM dataset
        tokenizer: RHM tokenizer instance
        config: Training configuration
        train_split_ratio: Ratio for train/eval split
        filter_config_L: Filter by hierarchy depth
        filter_config_m: Filter by multiplicity
        max_samples: Maximum samples to use

    Returns:
        Tuple of (train_dataset, eval_dataset, metadata)
    """
    print("=" * 60)
    print("PREPARING PACKED DATASET WITH CUSTOM TOKENIZER")
    print("=" * 60)

    # Import here to avoid circular dependencies
    from ICL.datasets.gen import UnifiedRHMDataset

    # Load unified dataset
    unified_dataset = UnifiedRHMDataset(dataset_path)
    dataset = unified_dataset.get_dataset()

    print(f"Original dataset size: {len(dataset):,}")

    # Apply filters if specified
    if filter_config_L is not None or filter_config_m is not None:
        dataset = unified_dataset.filter_by_config(L=filter_config_L, m=filter_config_m)
        print(f"After config filtering: {len(dataset):,}")

    # Limit samples if specified
    if max_samples is not None and len(dataset) > max_samples:
        indices = list(range(len(dataset)))
        random.shuffle(indices)
        dataset = dataset.select(indices[:max_samples])
        print(f"After sampling: {len(dataset):,}")

    # Prepare sequences for packing
    if config.pack_sequences:
        packed_dataset = _pack_sequences(dataset, tokenizer, config)
    else:
        packed_dataset = _prepare_individual_sequences(dataset, tokenizer, config)

    # Split into train/eval
    split_idx = int(len(packed_dataset) * train_split_ratio)
    train_dataset = packed_dataset.select(range(split_idx))
    eval_dataset = packed_dataset.select(range(split_idx, len(packed_dataset)))

    metadata = {
        "original_size": len(unified_dataset.get_dataset()),
        "filtered_size": len(dataset),
        "packed_size": len(packed_dataset),
        "train_size": len(train_dataset),
        "eval_size": len(eval_dataset),
        "packing_enabled": config.pack_sequences,
        "config": config,
        "vocab_info": unified_dataset.get_vocab_info(),
        "tokenizer_vocab_size": tokenizer.vocab_size,
    }

    print(f"Train dataset: {len(train_dataset):,}")
    print(f"Eval dataset: {len(eval_dataset):,}")
    print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
    print("=" * 60)

    return train_dataset, eval_dataset, metadata


def _pack_sequences(
    dataset: Dataset, tokenizer: RHMTokenizer, config: RHMTrainingConfig
) -> Dataset:
    """Pack multiple sequences into longer training examples."""
    print("Packing sequences with custom tokenizer...")

    packed_examples = []
    current_sequence = []
    current_length = 0

    # Reserve space for special tokens
    effective_max_length = config.max_sequence_length - 10

    for example in dataset:
        # Convert RHM sequence to token IDs using tokenizer
        if "input_ids" in example:
            sequence = example["input_ids"]
        elif "sequence" in example:
            sequence = tokenizer.encode_sequence(example["sequence"])
        else:
            continue

        # Skip empty sequences
        if not sequence:
            continue

        # If adding this sequence would exceed max length, finalize current packed sequence
        if current_length + len(sequence) + 1 > effective_max_length and current_sequence:
            # Add EOS token to end of packed sequence
            current_sequence.append(tokenizer.eos_token_id)
            packed_examples.append({
                "input_ids": current_sequence,
                "length": len(current_sequence),
            })
            current_sequence = []
            current_length = 0

        # Add separator if this isn't the first sequence in the pack
        if current_sequence:
            current_sequence.append(tokenizer.sep_token_id)
            current_length += 1

        # Add the sequence
        current_sequence.extend(sequence)
        current_length += len(sequence)

    # Add the last packed sequence if it exists
    if current_sequence:
        current_sequence.append(tokenizer.eos_token_id)
        packed_examples.append({
            "input_ids": current_sequence,
            "length": len(current_sequence),
        })

    print(f"Packed {len(dataset)} sequences into {len(packed_examples)} training examples")
    avg_length = sum(ex["length"] for ex in packed_examples) / len(packed_examples)
    print(f"Average packed sequence length: {avg_length:.1f}")

    return Dataset.from_list(packed_examples)


def _prepare_individual_sequences(
    dataset: Dataset, tokenizer: RHMTokenizer, config: RHMTrainingConfig
) -> Dataset:
    """Prepare individual sequences without packing."""
    print("Preparing individual sequences with custom tokenizer...")

    examples = []
    for example in dataset:
        # Convert RHM sequence to token IDs using tokenizer
        if "input_ids" in example:
            sequence = example["input_ids"]
        elif "sequence" in example:
            sequence = tokenizer.encode_sequence(example["sequence"])
        else:
            continue

        # Skip empty sequences
        if not sequence:
            continue

        # Add EOS token
        sequence.append(tokenizer.eos_token_id)

        # Truncate if too long
        if len(sequence) > config.max_sequence_length:
            sequence = sequence[:config.max_sequence_length]

        examples.append({
            "input_ids": sequence,
            "length": len(sequence),
        })

    print(f"Prepared {len(examples)} individual sequences")
    return Dataset.from_list(examples)

# train pipeline

In [ ]:
"""RHM Training Pipeline with Custom Tokenizer Support"""

import logging
import typing as t

import torch
from datasets import Dataset
from transformers import (
    DataCollatorForLanguageModeling,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

import os


# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RHMDataCollator(DataCollatorForLanguageModeling):
    """Data collator for RHM with custom tokenizer and separator token handling."""

    def __init__(
        self,
        tokenizer: RHMTokenizer,
        mlm: bool = True,
        mlm_probability: float = 0.15,
        return_tensors: str = "pt",
    ):
        """Initialize RHM data collator.

        Args:
            tokenizer: RHM tokenizer instance
            mlm: Whether to use masked language modeling
            mlm_probability: Probability of masking tokens
            return_tensors: Format of returned tensors
        """
        super().__init__(
            tokenizer=tokenizer,
            mlm=mlm,
            mlm_probability=mlm_probability,
            return_tensors=return_tensors,
        )
        self.rhm_tokenizer = tokenizer

    def torch_call(self, examples: list[dict[str, t.Any]]) -> dict[str, torch.Tensor]:
        """Process batch with RHM-specific token handling."""
        # Convert to format expected by parent class
        batch = []
        for example in examples:
            if isinstance(example["input_ids"], list):
                batch.append({"input_ids": torch.tensor(example["input_ids"])})
            else:
                batch.append({"input_ids": example["input_ids"]})

        # Use parent's processing for padding and masking
        result = super().torch_call(batch)

        # Apply RHM-specific masking rules
        if self.mlm and "labels" in result:
            # Never mask special tokens
            special_token_ids = {
                self.rhm_tokenizer.pad_token_id,
                self.rhm_tokenizer.eos_token_id,
                self.rhm_tokenizer.sep_token_id,
                self.rhm_tokenizer.unk_token_id,
            }
            
            for special_token_id in special_token_ids:
                special_mask = result["input_ids"] == special_token_id
                result["labels"][special_mask] = -100  # Don't compute loss on special tokens

        return result


class RHMTrainer(Trainer):
    """RHM trainer with custom tokenizer support and hierarchical metrics."""

    def __init__(
        self,
        model,
        args: TrainingArguments,
        train_dataset: Dataset,
        eval_dataset: Dataset | None = None,
        tokenizer: RHMTokenizer | None = None,
        data_collator: RHMDataCollator | None = None,
        config: RHMTrainingConfig | None = None,
        **kwargs,
    ):
        """Initialize RHM trainer.

        Args:
            model: The model to train
            args: Training arguments
            train_dataset: Training dataset
            eval_dataset: Evaluation dataset
            tokenizer: RHM tokenizer instance
            data_collator: Data collator
            config: RHM training configuration
            **kwargs: Additional arguments for Trainer
        """
        self.rhm_config = config
        self.rhm_tokenizer = tokenizer

        # Create default data collator if none provided
        if data_collator is None:
            if tokenizer is None:
                raise ValueError("Either tokenizer or data_collator must be provided")
            
            data_collator = RHMDataCollator(
                tokenizer=tokenizer,
                mlm=config.mlm if config else True,
                mlm_probability=config.mlm_probability if config else 0.15,
            )

        super().__init__(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=tokenizer,  # Pass tokenizer to parent
            data_collator=data_collator,
            **kwargs,
        )

    def compute_loss(self, model, inputs, return_outputs=False):
        """Compute loss with RHM-specific handling."""
        # Standard loss computation with special token masking handled by data collator
        return super().compute_loss(model, inputs, return_outputs)

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        """Enhanced evaluation with RHM-specific metrics."""
        # Standard evaluation
        results = super().evaluate(eval_dataset, ignore_keys, metric_key_prefix)
        
        # Add RHM-specific metrics if enabled
        if self.rhm_config and self.rhm_config.track_hierarchical_metrics:
            hierarchical_metrics = self._compute_hierarchical_metrics(eval_dataset)
            results.update(hierarchical_metrics)
        
        return results

    def _compute_hierarchical_metrics(self, eval_dataset=None) -> dict[str, float]:
        """Compute hierarchical-specific metrics."""
        # Placeholder for hierarchical metrics
        # In practice, you'd analyze model predictions on hierarchical patterns
        return {
            "eval_hierarchical_accuracy": 0.0,
            "eval_separator_token_accuracy": 0.0,
            "eval_special_token_perplexity": 0.0,
        }


class HierarchicalMetricsCallback(TrainerCallback):
    """Callback for computing hierarchical-specific metrics during training."""

    def __init__(self, eval_dataset: Dataset, config: RHMTrainingConfig, tokenizer: RHMTokenizer):
        """Initialize metrics callback.

        Args:
            eval_dataset: Evaluation dataset for computing metrics
            config: Training configuration
            tokenizer: RHM tokenizer instance
        """
        self.eval_dataset = eval_dataset
        self.config = config
        self.tokenizer = tokenizer
        self.step_count = 0

    def on_evaluate(self, args, state, control, model, logs=None, **kwargs):
        """Compute additional metrics during evaluation."""
        if logs is None:
            return

        self.step_count += 1
        
        # Add hierarchical evaluation metrics
        logs["hierarchical_eval_count"] = self.step_count
        
        # Example: Track special token usage
        if hasattr(model, 'get_input_embeddings'):
            embeddings = model.get_input_embeddings()
            special_token_norms = {}
            
            special_tokens = {
                'pad': self.tokenizer.pad_token_id,
                'eos': self.tokenizer.eos_token_id,
                'sep': self.tokenizer.sep_token_id,
                'mask': self.tokenizer.mask_token_id,
            }
            
            for token_name, token_id in special_tokens.items():
                if token_id < embeddings.num_embeddings:
                    norm = torch.norm(embeddings.weight[token_id]).item()
                    logs[f"special_token_{token_name}_norm"] = norm

        logger.info(f"Hierarchical evaluation #{self.step_count} completed")
        logger.info(f"Current eval loss: {logs.get('eval_loss', 'N/A'):.4f}")

    def on_train_begin(self, args, state, control, **kwargs):
        """Log tokenizer information at training start."""
        logger.info(f"Training with RHM tokenizer (vocab_size: {self.tokenizer.vocab_size})")
        logger.info(f"Special tokens - PAD: {self.tokenizer.pad_token_id}, "
                   f"EOS: {self.tokenizer.eos_token_id}, "
                   f"SEP: {self.tokenizer.sep_token_id}, "
                   f"MASK: {self.tokenizer.mask_token_id}")


def create_rhm_training_pipeline(
    dataset_path: str,
    model,
    training_config: RHMTrainingConfig,
    **dataset_kwargs,
) -> tuple[RHMTrainer, dict[str, t.Any]]:
    """Create complete RHM training pipeline with custom tokenizer.

    Args:
        dataset_path: Path to unified RHM dataset
        model: Model to train
        training_config: Training configuration
        **dataset_kwargs: Additional arguments for dataset preparation

    Returns:
        Tuple of (trainer, metadata)
    """
    print("Creating RHM training pipeline with custom tokenizer...")

    # Create tokenizer
    tokenizer = training_config.create_tokenizer()
    print(f"Created RHM tokenizer with vocab size: {tokenizer.vocab_size}")

    # Prepare datasets
    train_dataset, eval_dataset, metadata = prepare_packed_dataset(
        dataset_path=dataset_path,
        tokenizer=tokenizer,
        config=training_config,
        **dataset_kwargs
    )

    # Update model vocab size if needed
    if hasattr(model, 'resize_token_embeddings'):
        model.resize_token_embeddings(tokenizer.vocab_size)
        print(f"Resized model token embeddings to {tokenizer.vocab_size}")

    # Create training arguments
    training_args = training_config.to_training_arguments()

    # Create data collator
    data_collator = RHMDataCollator(
        tokenizer=tokenizer,
        mlm=training_config.mlm,
        mlm_probability=training_config.mlm_probability,
    )

    # Create callbacks
    callbacks = [HierarchicalMetricsCallback(eval_dataset, training_config, tokenizer)]

    # Create trainer
    trainer = RHMTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        config=training_config,
        callbacks=callbacks,
    )

    # Enhanced metadata
    enhanced_metadata = {
        "dataset_metadata": metadata,
        "tokenizer_metadata": {
            "vocab_size": tokenizer.vocab_size,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
            "sep_token_id": tokenizer.sep_token_id,
            "mask_token_id": tokenizer.mask_token_id,
            "special_tokens": {
                "pad": tokenizer.pad_token,
                "eos": tokenizer.eos_token,
                "sep": tokenizer.sep_token,
                "mask": tokenizer.mask_token,
                "unk": tokenizer.unk_token,
            }
        },
        "model_metadata": {
            "model_vocab_size": model.config.vocab_size if hasattr(model, 'config') else None,
            "hidden_size": model.config.hidden_size if hasattr(model, 'config') else None,
        }
    }

    print("Training pipeline created successfully!")
    print(f"Tokenizer: {tokenizer.__class__.__name__}")
    print(f"Data collator: {data_collator.__class__.__name__}")
    print(f"Trainer: {trainer.__class__.__name__}")
    
    return trainer, enhanced_metadata

/Users/jliu/anaconda3/lib/python3.11/site-packages/transformers/utils/generic.py:260: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


# test pipeline 

In [ ]:
#!/usr/bin/env python3
"""
RHM Model Training Main Script
Simple training pipeline with hardcoded defaults and optional YAML overrides
"""

import argparse
import logging
import typing as t
from pathlib import Path

import yaml
from transformers import set_seed

from ICL import settings


# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def parse_args1() -> argparse.Namespace:
    """Parse command line arguments."""
    parser = argparse.ArgumentParser(description="Train RHM models with hardcoded defaults and optional YAML overrides")
    parser.add_argument("--config", type=str, help="Path to YAML config file with parameter overrides (optional)")
    return parser.parse_args()


def parse_args() -> argparse.Namespace:
    """Parse command line arguments.""" 
    parser = argparse.ArgumentParser(description="Train RHM models with hardcoded defaults and optional YAML overrides")
    # Remove the argument parsing and hardcode the config path
    # parser.add_argument("--config", type=str, help="Path to YAML config file with parameter overrides (optional)")
    
    # Create a namespace with the hardcoded config path
    args = argparse.Namespace()
    args.config = "/scratch2/jliu/ICL/ICL_Modular_Arithmetic/src/scripts/model/conf/test.yaml"  # Replace with your actual config path
    return args


def load_yaml_overrides(config_path: str | Path | None) -> dict[str, t.Any]:
    """Load YAML configuration overrides if provided."""
    if config_path is None:
        logger.info("No YAML config provided, using hardcoded defaults")
        return {}

    config_path = Path(config_path)

    if not config_path.exists():
        logger.warning(f"Config file {config_path} not found. Using hardcoded defaults.")
        return {}

    try:
        with config_path.open("r", encoding="utf-8") as file:
            yaml_overrides = yaml.safe_load(file)

        if yaml_overrides:
            logger.info(f"Loaded YAML overrides from {config_path}")
            logger.info(f"YAML overrides: {yaml_overrides}")
            return yaml_overrides
        logger.warning(f"Config file {config_path} is empty. Using defaults.")
        return {}

    except yaml.YAMLError as e:
        logger.error(f"Error parsing YAML file {config_path}: {e}")
        logger.info("Falling back to hardcoded defaults")
        return {}
    except Exception as e:
        logger.error(f"Unexpected error loading config {config_path}: {e}")
        logger.info("Falling back to hardcoded defaults")
        return {}


def create_training_config(args: argparse.Namespace) -> RHMTrainingConfig:
    """Create training configuration with hardcoded defaults and YAML overrides."""
    # Start with hardcoded defaults from dataclass
    config = RHMTrainingConfig()
    logger.info("Using hardcoded defaults from RHMTrainingConfig")

    # Apply YAML overrides
    yaml_overrides = load_yaml_overrides(args.config)
    if yaml_overrides:
        # Convert dataclass to dict for merging
        config_dict = config.__dict__.copy()

        # Apply YAML overrides with type conversion
        for key, value in yaml_overrides.items():
            if hasattr(config, key):
                # Get the original type from the default config
                original_value = getattr(config, key)
                
                # Convert value to the correct type
                try:
                    if isinstance(original_value, bool):
                        converted_value = str(value).lower() in ['true', '1', 'yes', 'on']
                    elif isinstance(original_value, int):
                        converted_value = int(value)
                    elif isinstance(original_value, float):
                        converted_value = float(value)
                    elif isinstance(original_value, list):
                        converted_value = value if isinstance(value, list) else [value]
                    else:
                        converted_value = value
                    
                    config_dict[key] = converted_value
                    logger.info(f"YAML override: {key} = {converted_value} (type: {type(converted_value).__name__})")
                    
                except (ValueError, TypeError) as e:
                    logger.warning(f"Failed to convert {key}={value}: {e}. Using as string.")
                    config_dict[key] = value
            else:
                logger.warning(f"Unknown parameter in YAML: {key} (ignored)")

        # Create new config with overrides
        config = RHMTrainingConfig(**config_dict)

    return config


def main() -> dict:
    """Train RHM model using simplified configuration system."""
    # Parse arguments and create configuration
    args = parse_args()
    config = create_training_config(args)
    # Set seed for reproducibility
    set_seed(config.seed)

    # Log final configuration
    logger.info("Final training configuration:")
    logger.info(f"  Task: {config.task_name}")
    logger.info(f"  Learning rate: {config.learning_rate}")
    logger.info(f"  Epochs: {config.num_train_epochs}")
    logger.info(f"  Train batch size: {config.per_device_train_batch_size}")
    logger.info(f"  Eval batch size: {config.per_device_eval_batch_size}")
    logger.info(f"  Vocab size: {config.vocab_size} (effective: {config.effective_vocab_size})")
    logger.info(f"  Max sequence length: {config.max_sequence_length}")
    logger.info(f"  Pack sequences: {config.pack_sequences}")
    logger.info(f"  Output dir: {config.output_dir}")
    logger.info(f"  Run name: {config.run_name}")

    # Set dataset path (hardcoded default with YAML override option)
    dataset_path = getattr(config, 'dataset_path', settings.PATH.train_dir / "raw")
    if hasattr(config, 'dataset_path'):
        dataset_path = config.dataset_path
    else:
        dataset_path = settings.PATH.train_dir / "raw"
        logger.info(f"Using default dataset path: {dataset_path}")

    # Create model based on configuration
    logger.info("Creating model...")
    if config.model_name_or_path:
        # Load from existing model
        from transformers import AutoConfig, AutoModelForCausalLM, AutoModelForMaskedLM
        
        model_config = AutoConfig.from_pretrained(config.model_name_or_path)
        model_config.vocab_size = config.effective_vocab_size
        
        if config.task_name == "clm":
            model = AutoModelForCausalLM.from_pretrained(config.model_name_or_path, config=model_config)
        else:
            model = AutoModelForMaskedLM.from_pretrained(config.model_name_or_path, config=model_config)
    else:
        # Create new model
        if config.task_name == "clm":
            from transformers import GPT2Config, AutoModelForCausalLM
            
            model_config = GPT2Config(
                vocab_size=config.effective_vocab_size,
                n_positions=config.max_position_embeddings,
                n_embd=config.hidden_size,
                n_layer=config.num_hidden_layers,
                n_head=config.num_attention_heads,
                n_inner=config.intermediate_size,
                resid_pdrop=0.1,
                embd_pdrop=0.1,
                attn_pdrop=0.1,
                use_cache=False,
                pad_token_id=config.pad_token_id,
            )
            model = AutoModelForCausalLM.from_config(model_config)
            
        elif config.task_name == "mlm":
            from transformers import BertConfig, AutoModelForMaskedLM
            
            model_config = BertConfig(
                vocab_size=config.effective_vocab_size,
                hidden_size=config.hidden_size,
                num_hidden_layers=config.num_hidden_layers,
                num_attention_heads=config.num_attention_heads,
                intermediate_size=config.intermediate_size,
                max_position_embeddings=config.max_position_embeddings,
                hidden_dropout_prob=0.1,
                attention_probs_dropout_prob=0.1,
                pad_token_id=config.pad_token_id,
                mask_token_id=config.mask_token_id,
                cls_token_id=config.cls_token_id,
                sep_token_id=config.sep_token_id,
            )
            model = AutoModelForMaskedLM.from_config(model_config)
        else:
            raise ValueError(f"Unknown task: {config.task_name}")
    
    logger.info(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

    # Create trainer using your existing pipeline function
    logger.info("Creating training pipeline...")


    trainer, metadata = create_rhm_training_pipeline(
        dataset_path=str(dataset_path),
        model=model,
        training_config=config,  # Use 'training_config' instead of 'config'
        train_split_ratio=getattr(config, 'train_split_ratio', 0.8),
        filter_config_L=getattr(config, 'filter_config_L', None),
        filter_config_m=getattr(config, 'filter_config_m', None),
        max_samples=getattr(config, 'max_samples', None),
    )

    # Log dataset info
    logger.info("Dataset preparation completed:")
    #logger.info(f"Starting training for {config.task_name.upper()} task...")
    results = trainer.train()
    trainer.save_model()  # This saves to output_dir
    trainer.save_state()  # This saves trainer state
    logger.info(f"Training completed! Results saved to {config.output_dir}")

    return results


if __name__ == "__main__":
    main()

/Users/jliu/workspace/ICL/ICL_Modular_Arithmetic/src/ICL/settings.py:28: UserWarning: Provided DATA_DIR: /Users/jliu/local_data does not exist.
Set $DATA_DIR or check hostname defaults.
  _warnings.warn(
INFO:__main__:Using hardcoded defaults from RHMTrainingConfig
INFO:__main__:Loaded YAML overrides from /Users/jliu/workspace/ICL/ICL_Modular_Arithmetic/src/scripts/model/conf/test.yaml
INFO:__main__:YAML overrides: {'dataset_path': '/Users/jliu/workspace/ICL/datasets/train/raw', 'train_split_ratio': 0.8, 'filter_config_L': 3, 'filter_config_m': 2, 'max_samples': 10000, 'task_name': 'clm', 'vocab_size': 32, 'hidden_size': 512, 'num_hidden_layers': 6, 'num_attention_heads': 8, 'intermediate_size': 2048, 'max_position_embeddings': 2048, 'num_train_epochs': 5, 'per_device_train_batch_size': 16, 'per_device_eval_batch_size': 32, 'gradient_accumulation_steps': 1, 'learning_rate': '5e-4', 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'pack_sequences': True, 'max_sequence_length': 512, 'separator

Creating RHM training pipeline with custom tokenizer...
Created RHM tokenizer with vocab size: 37
PREPARING PACKED DATASET WITH CUSTOM TOKENIZER
Loading RHM dataset...
✓ Loaded dataset with 5000 sequences
✓ Loaded metadata for 5 configurations
Validating dataset structure...
✓ Dataset validation passed
Computing dataset statistics...
✓ Statistics computed
Original dataset size: 5,000


You are resizing the embedding layer without providing a `pad_to_multiple_of` parameter. This means that the new embeding dimension will be 37. This might induce some performance reduction as *Tensor Cores* will not be available. For more details  about this, or help on choosing the correct value for resizing, refer to this guide: https://docs.nvidia.com/deeplearning/performance/dl-performance-matrix-multiplication/index.html#requirements-tc


After config filtering: 1,000
Packing sequences with custom tokenizer...
Packed 1000 sequences into 19 training examples
Average packed sequence length: 473.7
Train dataset: 15
Eval dataset: 4
Tokenizer vocab size: 37
Resized model token embeddings to 37


TypeError: Accelerator.__init__() got an unexpected keyword argument 'dispatch_batches'